In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy import stats

# ============================================================================
# DATASET: EIT / EMG / CAP / ANGLE
# ============================================================================

class CrossModalForceDataset(Dataset):
    """
    Loads EIT, EMG, CAP, ANGLE windows from the normalized CSV.

    - EIT: 4 channels  -> ['eit1','eit2','eit3','eit4']
    - EMG: 1 channel   -> ['emg_ch0']
    - CAP: 1 channel   -> 'cap_pf' / 'cap_value' / 'capacitance' (auto-detect)
    - ANGLE: 1 channel -> 'angle' / 'angle_deg' / 'angle_rad' (auto-detect)
    """
    def __init__(self, csv_path, segment_ids=None, seq_length=50, stride=10):
        self.seq_length = seq_length
        df = pd.read_csv(csv_path)

        if segment_ids is not None:
            df = df[df['segment_id'].isin(segment_ids)]

        # --- auto-detect CAP column ---
        cap_candidates = ['cap_pf', 'cap_value', 'capacitance']
        cap_col = None
        for c in cap_candidates:
            if c in df.columns:
                cap_col = c
                break
        if cap_col is None:
            raise ValueError(
                f"No CAP column found. Tried: {cap_candidates}. "
                f"Available columns: {list(df.columns)}"
            )

        # --- auto-detect ANGLE column ---
        angle_candidates = ['angle', 'angle_deg', 'angle_rad']
        angle_col = None
        for c in angle_candidates:
            if c in df.columns:
                angle_col = c
                break
        if angle_col is None:
            raise ValueError(
                f"No ANGLE column found. Tried: {angle_candidates}. "
                f"Available columns: {list(df.columns)}"
            )

        self.sequences = []

        for seg_id in df['segment_id'].unique():
            seg_data = df[df['segment_id'] == seg_id].sort_values('time_sec')

            eit   = seg_data[['eit1', 'eit2', 'eit3', 'eit4']].values
            emg   = seg_data[['emg_ch0']].values
            cap   = seg_data[[cap_col]].values
            angle = seg_data[[angle_col]].values

            num_windows = (len(seg_data) - seq_length) // stride + 1

            for i in range(num_windows):
                start_idx = i * stride
                end_idx   = start_idx + seq_length

                if end_idx <= len(seg_data):
                    self.sequences.append({
                        'eit':   eit[start_idx:end_idx].astype(np.float32),   # [T, 4]
                        'emg':   emg[start_idx:end_idx].astype(np.float32),   # [T, 1]
                        'cap':   cap[start_idx:end_idx].astype(np.float32),   # [T, 1]
                        'angle': angle[start_idx:end_idx].astype(np.float32)  # [T, 1]
                    })

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        return {
            'eit':   torch.FloatTensor(seq['eit']),     # [T, 4]
            'emg':   torch.FloatTensor(seq['emg']),     # [T, 1]
            'cap':   torch.FloatTensor(seq['cap']),     # [T, 1]
            'angle': torch.FloatTensor(seq['angle'])    # [T, 1]
        }
# ============================================================================
# ENCODERS
# ============================================================================

class EITEncoder(nn.Module):
    def __init__(self, input_size=4, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

    def forward(self, x):
        # x: [B, T, 4]
        _, (hidden, _) = self.lstm(x)
        return hidden[-1]  # [B, hidden_size]


class EMGEncoder(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

    def forward(self, x):
        # x: [B, T, 1]
        _, (hidden, _) = self.lstm(x)
        return hidden[-1]  # [B, hidden_size]


class CapEncoder(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

    def forward(self, x):
        # x: [B, T, 1]
        _, (hidden, _) = self.lstm(x)
        return hidden[-1]  # [B, hidden_size]


# ============================================================================
# FLEXIBLE FUSION (1–3 modalities)
# ============================================================================

class FusionFlexible(nn.Module):
    """
    Takes a list of encoder features and fuses them by concatenation.
    number_of_modalities can be 1, 2, or 3.
    """
    def __init__(self, encoder_hidden=64, shared_size=128, dropout=0.3):
        super().__init__()
        self.fc = nn.Linear(encoder_hidden * 3, shared_size)  # we'll mask unused dims
        self.bn = nn.BatchNorm1d(shared_size)
        self.dropout = nn.Dropout(dropout)
        self.encoder_hidden = encoder_hidden

    def forward(self, feature_list):
        """
        feature_list: list of [B, encoder_hidden], length = 1, 2, or 3.
        We pack them into a fixed-size vector of length 3*encoder_hidden:
        [feat1, feat2, feat3 or zeros]
        """
        B = feature_list[0].shape[0]
        device = feature_list[0].device

        # initialize zeros: [B, 3 * encoder_hidden]
        fused = torch.zeros(B, 3 * self.encoder_hidden, device=device)

        for i, feat in enumerate(feature_list):
            start = i * self.encoder_hidden
            end   = (i + 1) * self.encoder_hidden
            fused[:, start:end] = feat

        x = self.fc(fused)
        x = self.bn(x)
        x = torch.relu(x)
        x = self.dropout(x)
        return x  # [B, shared_size]

# ============================================================================
# FLEXIBLE MULTI-MODAL → ANGLE MODEL
# ============================================================================

class FlexibleMultiModalToAngleModel(nn.Module):
    """
    Predicts ANGLE from a configurable set of modalities:
    - input_modalities is a list: subset of ['eit', 'emg', 'cap']
      e.g., ['eit', 'emg', 'cap'], ['eit', 'emg'], ['cap'], etc.
    """
    def __init__(self,
                 input_modalities=('eit', 'emg', 'cap'),
                 seq_length=50,
                 encoder_hidden=64,
                 encoder_layers=2,
                 shared_size=128,
                 decoder_hidden=64,
                 decoder_layers=2,
                 dropout=0.3):
        super().__init__()

        self.input_modalities = list(input_modalities)
        self.seq_length = seq_length

        # encoders
        self.eit_encoder = EITEncoder(
            input_size=4,
            hidden_size=encoder_hidden,
            num_layers=encoder_layers,
            dropout=dropout
        )
        self.emg_encoder = EMGEncoder(
            input_size=1,
            hidden_size=encoder_hidden,
            num_layers=encoder_layers,
            dropout=dropout
        )
        self.cap_encoder = CapEncoder(
            input_size=1,
            hidden_size=encoder_hidden,
            num_layers=encoder_layers,
            dropout=dropout
        )

        # fusion
        self.fusion = FusionFlexible(
            encoder_hidden=encoder_hidden,
            shared_size=shared_size,
            dropout=dropout
        )

        # decoder
        self.angle_decoder = AngleDecoder(
            input_size=shared_size,
            hidden_size=decoder_hidden,
            seq_length=seq_length,
            num_layers=decoder_layers,
            dropout=dropout
        )

    def forward(self, eit=None, emg=None, cap=None):
        features = []

        if 'eit' in self.input_modalities:
            if eit is None:
                raise ValueError("EIT modality requested but eit=None passed to forward().")
            features.append(self.eit_encoder(eit))   # [B, H]

        if 'emg' in self.input_modalities:
            if emg is None:
                raise ValueError("EMG modality requested but emg=None passed to forward().")
            features.append(self.emg_encoder(emg))   # [B, H]

        if 'cap' in self.input_modalities:
            if cap is None:
                raise ValueError("CAP modality requested but cap=None passed to forward().")
            features.append(self.cap_encoder(cap))   # [B, H]

        if len(features) == 0:
            raise ValueError("No modalities specified in input_modalities.")

        shared_feat = self.fusion(features)          # [B, shared_size]
        angle_pred = self.angle_decoder(shared_feat) # [B, T, 1]

        return angle_pred
# ============================================================================
# TRAIN / VAL / EVAL FOR ANGLE
# ============================================================================

def train_epoch_angle(model, dataloader, optimizer, device):
    model.train()
    criterion = nn.MSELoss()
    losses = []

    for batch in tqdm(dataloader, desc='Training (→ ANGLE)'):
        eit   = batch['eit'].to(device)
        emg   = batch['emg'].to(device)
        cap   = batch['cap'].to(device)
        angle = batch['angle'].to(device)

        # model will only use modalities in model.input_modalities
        angle_pred = model(eit=eit, emg=emg, cap=cap)

        loss = criterion(angle_pred, angle)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        losses.append(loss.item())

    return np.mean(losses) if losses else 0.0


def validate_angle(model, dataloader, device):
    model.eval()
    criterion = nn.MSELoss()
    losses = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Validating (→ ANGLE)'):
            eit   = batch['eit'].to(device)
            emg   = batch['emg'].to(device)
            cap   = batch['cap'].to(device)
            angle = batch['angle'].to(device)

            angle_pred = model(eit=eit, emg=emg, cap=cap)
            loss = criterion(angle_pred, angle)
            losses.append(loss.item())

    return np.mean(losses) if losses else 0.0


def evaluate_angle(model, dataloader, device):
    model.eval()
    preds = []
    gts   = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Evaluating ANGLE'):
            eit   = batch['eit'].to(device)
            emg   = batch['emg'].to(device)
            cap   = batch['cap'].to(device)
            angle = batch['angle'].to(device)

            angle_pred = model(eit=eit, emg=emg, cap=cap)

            preds.append(angle_pred.cpu().numpy())
            gts.append(angle.cpu().numpy())

    preds = np.concatenate(preds, axis=0)  # [N, T, 1]
    gts   = np.concatenate(gts, axis=0)    # [N, T, 1]

    pred_flat = preds.reshape(-1)
    true_flat = gts.reshape(-1)

    mse  = mean_squared_error(true_flat, pred_flat)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(true_flat, pred_flat)
    corr, _ = stats.pearsonr(pred_flat, true_flat)

    ss_res = np.sum((true_flat - pred_flat) ** 2)
    ss_tot = np.sum((true_flat - true_flat.mean()) ** 2)
    r2 = 1 - (ss_res / ss_tot)

    metrics = {
        'mse': mse,
        'rmse': rmse,
        'mae': mae,
        'correlation': corr,
        'r2': r2
    }

    return preds, gts, metrics
INPUT_MODALITIES = ['eit', 'emg', 'cap']   # use all 3

seq_length   = 30
batch_size   = 32
num_epochs   = 200
learning_rate = 1e-3
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# split segments
DATA_PATH = '/content/drive/MyDrive/Multi_Modal_Rehab/data_normalized_all.csv'
df = pd.read_csv(DATA_PATH)
all_segments = sorted(df['segment_id'].unique())
np.random.seed(42)
shuffled = np.random.permutation(all_segments)

n_train = int(len(all_segments) * 0.6)
n_val   = int(len(all_segments) * 0.2)

train_segs = shuffled[:n_train].tolist()
val_segs   = shuffled[n_train:n_train + n_val].tolist()
test_segs  = shuffled[n_train + n_val:].tolist()

train_dataset = CrossModalAngleDataset(DATA_PATH, train_segs, seq_length, stride=2)
val_dataset   = CrossModalAngleDataset(DATA_PATH, val_segs,   seq_length, stride=2)
test_dataset  = CrossModalAngleDataset(DATA_PATH, test_segs,  seq_length, stride=2)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

model = FlexibleMultiModalToAngleModel(
    input_modalities=INPUT_MODALITIES,
    seq_length=seq_length,
    encoder_hidden=64,
    encoder_layers=2,
    shared_size=128,
    decoder_hidden=64,
    decoder_layers=2,
    dropout=0.3
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)

best_val = float('inf')
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")
    train_loss = train_epoch_angle(model, train_loader, optimizer, device)
    val_loss   = validate_angle(model, val_loader, device)
    print(f"  Train: {train_loss:.4f} | Val: {val_loss:.4f}")

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), 'best_angle_model.pth')
        print("  ✓ Best model saved!")

print("Best val loss:", best_val)

# eval
model.load_state_dict(torch.load('best_angle_model.pth', map_location=device))
preds, gts, metrics = evaluate_angle(model, test_loader, device)
print("ANGLE metrics:", metrics)